# 🧩 Mini-Lab: Typed Shapes — Hints and Dataclasses

**Module 2: The Python That Runs Your Pipeline** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Annotate** parameters and returns with type hints a checker can verify
2. **Predict** which snippets a type checker flags
3. **Convert** loose dicts into `Message` and `TriageResult` dataclasses

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Type hints | Documentation the machine can check; no runtime behaviour change |
| Dataclass as a shape | Function for behaviour, dataclass for shape, class only for both |


## Why Type Hints Matter for AI Engineering

In the previous mini-lab, we extracted pure functions — all inputs as parameters, all output as a return value. Type hints take this one step further: they **document the contract** of each parameter and the return value so that:

- Your **IDE** auto-completes correctly and warns you about wrong types *as you type*.
- A **type checker** (mypy, Pyright) catches mismatches before you run anything.
- A **teammate** reading your function signature knows exactly what to pass and what they'll get back — no guessing.

Type hints are *optional* at runtime — Python won't enforce them. But tooling will, and that's where the bugs get caught.

## Before and After: Adding Type Hints

Let's take the `score_documents` function from the previous mini-lab and add type hints to it.

### Without type hints

The un-annotated version works, but the signature tells you nothing about types:

In [1]:
# Un-annotated — what types does this expect? What does it return?
def score_documents_untyped(query, documents, threshold=0.3):
    """Score documents by keyword overlap with the query."""
    query_words = set(query.lower().split())
    results = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words) / len(query_words)
        if overlap >= threshold:
            results.append({"text": doc, "score": round(overlap, 2)})
    results.sort(key=lambda x: x["score"], reverse=True)
    return results

Looking at the signature `(query, documents, threshold=0.3)`, you'd have to *read the body* to figure out:
- `query` is a string
- `documents` is a list of strings
- `threshold` is a float
- It returns a list of dicts

### With type hints

Now the signature tells the full story at a glance:

In [2]:
def score_documents(query: str, documents: list[str], threshold: float = 0.3) -> list[dict]:
    """Score documents by keyword overlap with the query."""
    query_words = set(query.lower().split())
    results = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words) / len(query_words)
        if overlap >= threshold:
            results.append({"text": doc, "score": round(overlap, 2)})
    results.sort(key=lambda x: x["score"], reverse=True)
    return results

In [3]:
# Quick test — same behavior, but now the contract is explicit
docs = [
    "Transformers use attention to process sequences in parallel.",
    "RNNs handle sequences one token at a time.",
    "Positional encoding tells transformers about word order.",
]

results = score_documents("How do transformers handle long sequences?", docs)
for r in results:
    print(f"  [{r['score']}] {r['text']}")

The code is identical — only the signature changed. But now any IDE or type checker knows exactly what goes in and what comes out.

## Common Type Hint Patterns

Here are the patterns you'll use most often in AI engineering code:

| Pattern | Example | When to Use |
|---------|---------|-------------|
| Basic types | `str`, `int`, `float`, `bool` | Single values |
| Lists | `list[str]` | A list of items of one type |
| Dicts | `dict[str, float]` | Key-value mappings |
| Optional | `str \| None` | A value that might be `None` |
| Union | `str \| int` | Could be either type |
| Return None | `-> None` | Function that returns nothing |

> **Note:** The `str | None` syntax requires Python 3.10+. For older versions, use `Optional[str]` from the `typing` module.

## Patterns in Action

Let's see these patterns in small, realistic functions you'd write in an AI pipeline:

In [4]:
def build_request(prompt: str, model: str, max_tokens: int = 500) -> dict[str, object]:
    """Build an LLM API request dict."""
    return {
        "model": model,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }


def find_best_chunk(scores: list[float]) -> int | None:
    """Return the index of the highest score, or None if the list is empty."""
    if not scores:
        return None
    return scores.index(max(scores))


def format_sources(docs: list[dict[str, str]], max_sources: int = 3) -> str:
    """Format retrieved documents into a numbered source list."""
    lines = []
    for i, doc in enumerate(docs[:max_sources], 1):
        lines.append(f"{i}. {doc['text']}")
    return "\n".join(lines)

In [5]:
# Test each function
print("build_request:")
print(build_request("Explain RAG", "gpt-4o-mini"))

print("\nfind_best_chunk:")
print(find_best_chunk([0.4, 0.9, 0.7]))   # → 1
print(find_best_chunk([]))                  # → None

print("\nformat_sources:")
sample_docs = [{"text": "Doc A"}, {"text": "Doc B"}, {"text": "Doc C"}, {"text": "Doc D"}]
print(format_sources(sample_docs, max_sources=2))

build_request:
{'model': 'gpt-4o-mini', 'max_tokens': 500, 'messages': [{'role': 'user', 'content': 'Explain RAG'}]}

find_best_chunk:
1
None

format_sources:
1. Doc A
2. Doc B


## How Type Hints Catch Bugs

Type hints don't enforce anything at runtime — Python happily runs mismatched types until something crashes. The value is that **tooling catches the bug earlier**.

Let's see an example of a bug that type hints would flag:

In [6]:
# This call has a bug: threshold should be a float, not a string
# A type checker would flag this BEFORE you run it.

# score_documents("transformers", docs, threshold="high")  # ← TypeError at runtime!

# Without type hints, you'd only discover this when the comparison
# `overlap >= threshold` fails at runtime (str vs float).
# With type hints, your IDE underlines "high" immediately.

print('Bug example: score_documents(..., threshold="high")')
print("  → Without hints: crashes at runtime with TypeError")
print("  → With hints: IDE/checker flags it before you run")

Bug example: score_documents(..., threshold="high")
  → Without hints: crashes at runtime with TypeError
  → With hints: IDE/checker flags it before you run


## Reading a Typed Signature

You can always inspect a function's type hints programmatically with `get_type_hints` — useful for understanding code you didn't write:

In [7]:
from typing import get_type_hints
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

hints = get_type_hints(score_documents)

lines = ["| Parameter | Type |", "|-----------|------|"] 
for name, hint in hints.items():
    lines.append(f"| `{name}` | `{hint}` |")

md("### Type hints for `score_documents`\n\n" + "\n".join(lines))

### Type hints for `score_documents`

| Parameter | Type |
|-----------|------|
| `query` | `<class 'str'>` |
| `documents` | `list[str]` |
| `threshold` | `<class 'float'>` |
| `return` | `list[dict]` |

The `return` key in the hints table shows the return type — this is what `-> list[dict]` produces.

This is exactly the information your IDE uses to auto-complete and warn about type mismatches.

---
## Dataclasses as shapes

Type hints document the function. A dataclass documents the record.


## The Problem: Dicts Everywhere

In the previous mini-labs, our `score_documents` function returns `list[dict]`. That works — but what keys does each dict have? What types are the values? You have to read the function body to find out:

```python
results.append({"text": doc, "score": round(overlap, 2)})
```

A dict is **shapeless** — Python doesn't stop you from writing `result["txt"]` (typo) or `result["score"] + result["text"]` (type mismatch). You only find out at runtime.

A **dataclass** gives this bundle of values a name and typed fields so tooling catches mistakes before you run anything.

## Dict vs Dataclass: Side by Side

In [1]:
# --- Dict approach: no structure enforced ---
result_dict = {"text": "Transformers use attention.", "score": 0.71}

# Typo goes unnoticed until runtime
# result_dict["txt"]  # ← KeyError, but nothing warns you beforehand

print("Dict:", result_dict)
print("Access:", result_dict["score"])

Dict: {'text': 'Transformers use attention.', 'score': 0.71}
Access: 0.71


In [2]:
from dataclasses import dataclass


@dataclass
class ScoredDoc:
    """A document with its relevance score."""
    text: str
    score: float


# --- Dataclass approach: named, typed fields ---
result_dc = ScoredDoc(text="Transformers use attention.", score=0.71)

# result_dc.txt  # ← IDE underlines this immediately (AttributeError)

print("Dataclass:", result_dc)
print("Access:   ", result_dc.score)

Dataclass: ScoredDoc(text='Transformers use attention.', score=0.71)
Access:    0.71


Notice what `@dataclass` gives you **for free**:

| Feature | Dict | Dataclass |
|---------|------|-----------|
| Named fields with types | ❌ | ✅ |
| Auto `__init__` | N/A | ✅ |
| Auto `__repr__` | ❌ (prints raw dict) | ✅ (prints class name + fields) |
| IDE auto-complete on fields | ❌ | ✅ |
| Typo caught before runtime | ❌ | ✅ |

## Using a Dataclass in a Function

Let's refactor `score_documents` to return `list[ScoredDoc]` instead of `list[dict]`. The logic is identical — only the container changes.

In [3]:
def score_documents(query: str, documents: list[str], threshold: float = 0.3) -> list[ScoredDoc]:
    """Score documents by keyword overlap. Returns ScoredDoc instances."""
    query_words = set(query.lower().split())
    results = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words) / len(query_words)
        if overlap >= threshold:
            results.append(ScoredDoc(text=doc, score=round(overlap, 2)))
    results.sort(key=lambda x: x.score, reverse=True)
    return results


docs = [
    "Transformers use attention to process sequences in parallel.",
    "RNNs handle sequences one token at a time.",
    "Positional encoding tells transformers about word order.",
]

results = score_documents("How do transformers handle long sequences?", docs)
for r in results:
    print(f"  [{r.score}] {r.text}")

The return type `list[ScoredDoc]` tells every caller exactly what fields exist — no guessing, no reading the function body.

## Dataclass Features: Defaults and Frozen

Dataclasses support default values and immutability — two things you'll use often.

In [4]:
@dataclass
class LLMRequest:
    """Configuration for an LLM API call."""
    prompt: str
    model: str = "gpt-4o-mini"
    max_tokens: int = 500
    temperature: float = 0.0


# Only the required field needs to be specified
req1 = LLMRequest(prompt="Explain RAG in one sentence.")
print("Defaults:", req1)

# Override any default
req2 = LLMRequest(prompt="Write a poem.", model="gpt-4o", temperature=0.9)
print("Custom: ", req2)

Defaults: LLMRequest(prompt='Explain RAG in one sentence.', model='gpt-4o-mini', max_tokens=500, temperature=0.0)
Custom:  LLMRequest(prompt='Write a poem.', model='gpt-4o', max_tokens=500, temperature=0.9)


In [5]:
# frozen=True makes instances immutable — good for configs you don't want mutated
@dataclass(frozen=True)
class ChunkConfig:
    """Immutable chunking configuration."""
    chunk_size: int = 500
    overlap: int = 50


cfg = ChunkConfig(chunk_size=300)
print(cfg)

# Trying to mutate raises an error
try:
    cfg.chunk_size = 1000
except AttributeError as e:
    print(f"Mutation blocked: {e}")

ChunkConfig(chunk_size=300, overlap=50)
Mutation blocked: cannot assign to field 'chunk_size'


## When Does a Full Class Earn Its Place?

A dataclass is a **shape** — it holds data. A full class adds **behavior** (methods that do non-trivial work, manage internal state, or control a lifecycle like open/close).

Here's the decision rule:

| Situation | Use |
|-----------|-----|
| A bundle of related values passed around | **Dataclass** |
| A throwaway one-off with 1–2 keys | **Dict** (it's fine) |
| Logic that transforms inputs → outputs | **Pure function** |
| An object that manages state or resources (DB connection, client session) | **Full class** |

Let's see a concrete example where a full class earns its place.

In [6]:
# A full class earns its place when it manages state or resources.
# This token budget tracker has internal state that changes over its lifetime.

class TokenBudget:
    """Tracks token usage against a budget."""

    def __init__(self, limit: int) -> None:
        self.limit = limit
        self._used = 0

    def spend(self, tokens: int) -> None:
        self._used += tokens

    @property
    def remaining(self) -> int:
        return max(0, self.limit - self._used)

    def __repr__(self) -> str:
        return f"TokenBudget(used={self._used}, remaining={self.remaining}, limit={self.limit})"


budget = TokenBudget(limit=1000)
budget.spend(350)
budget.spend(200)
print(budget)
print(f"Remaining: {budget.remaining}")

TokenBudget(used=550, remaining=450, limit=1000)
Remaining: 450


`TokenBudget` has **mutable internal state** (`_used` changes over time) and **behavior** (`spend`, `remaining`). A dataclass wouldn't be the right fit here because the object isn't just holding data — it's *managing* it.

In contrast, `ScoredDoc` and `LLMRequest` are just shapes — they hold values and that's it. Dataclass is the perfect tool for those.

## Quick Decision Checklist

When you're about to create a new container for data, ask these questions in order:

1. **Is it a one-off with 1–2 keys you'll use once?** → Use a **dict**. Don't over-engineer it.
2. **Is it a named bundle of fields passed between functions?** → Use a **dataclass**.
3. **Does it need to manage changing internal state or resources?** → Use a **full class**.
4. **Is it pure logic that transforms inputs to outputs?** → Use a **function** — you don't need a container at all.

## 🎯 Summary

### Key Takeaways

1. **Type hints annotate the contract** — they declare what a function expects and what it returns, making pure functions self-documenting
2. **Common patterns are simple** — `str`, `list[str]`, `dict[str, float]`, `int | None` cover most AI engineering code
3. **Bugs get caught earlier** — your IDE and type checkers (mypy, Pyright) flag type mismatches before runtime, which is far cheaper than debugging a crashed pipeline
4. **Zero runtime cost** — Python ignores type hints at runtime; they exist purely for tooling and human readers